# RM2 Sentiment Analysis Final

Notebook ini adalah entrypoint kanonis untuk pelaporan dan visualisasi final RM2 Sentiment. Notebook ini **tidak** melatih ulang model, tidak memilih threshold, tidak membuka ulang locked test, dan tidak menjalankan full inference. Semua tabel dan gambar dibaca/dibuat dari artefak final yang sudah dibekukan di `output/rm2_sentiment/final/`.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "dataset.csv").exists() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Project root tidak ditemukan.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.project_paths import (
    RM2_SENTIMENT_FINAL_DIR,
    RM2_SENTIMENT_FINAL_TABLES_DIR,
    RM2_SENTIMENT_FINAL_VISUALIZATION_DIR,
    RM2_SENTIMENT_MODEL_DIR,
    relative_to_root,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Final sentiment output: {relative_to_root(RM2_SENTIMENT_FINAL_DIR)}")

Project root: D:\UNHAS\PKM-RSH\Pendanaan\Analisis 4
Final sentiment output: output/rm2_sentiment/final


In [2]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

plt.style.use("seaborn-v0_8-whitegrid")

VIS_DIR = RM2_SENTIMENT_FINAL_VISUALIZATION_DIR
VIS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = ["Positive", "Neutral", "Negative", "Uncertain", "No Text"]
EVALUABLE_LABELS = ["Positive", "Neutral", "Negative"]
COLORS = {
    "Positive": "#2E7D32",
    "Neutral": "#8F8F8F",
    "Negative": "#C62828",
    "Uncertain": "#D6A21E",
    "No Text": "#CFCFCF",
}

def pct(value: float) -> str:
    return f"{value * 100:.1f}%"

## Load Final Artifacts

Seluruh data berikut adalah output final yang sudah tersedia. Notebook hanya membaca tabel ini dan membuat visualisasi presentasi.

In [3]:
summary = pd.read_csv(RM2_SENTIMENT_FINAL_DIR / "FINAL_SENTIMENT_ANALYSIS_SUMMARY.csv")
distribution = pd.read_csv(RM2_SENTIMENT_FINAL_TABLES_DIR / "sentiment_distribution_observational.csv")
hcc_vs_nonhcc = pd.read_csv(RM2_SENTIMENT_FINAL_TABLES_DIR / "hcc_vs_nonhcc_comment_sentiment_v2.csv")
hcc_goals = pd.read_csv(RM2_SENTIMENT_FINAL_TABLES_DIR / "hcc_sentiment_goals_summary_v2.csv")
confusion = pd.read_csv(RM2_SENTIMENT_MODEL_DIR / "final_locked_test_confusion_matrix.csv")
metrics = pd.read_csv(RM2_SENTIMENT_MODEL_DIR / "final_locked_test_metrics.csv")
lock = json.loads((RM2_SENTIMENT_MODEL_DIR / "final_locked_test_evaluation_lock.json").read_text(encoding="utf-8"))

summary_map = dict(zip(summary["metric"], summary["value"]))
expected_counts = {
    "final_acceptance_status": "FINAL_MODEL_VALIDATED",
    "observational_count": "33063",
    "injected_diagnostic_count": "784",
    "hcc_count": "42",
    "hcc_member_count": "218",
    "observational_Positive": "2718",
    "observational_Neutral": "23977",
    "observational_Negative": "4771",
    "observational_Uncertain": "1593",
    "observational_No Text": "4",
}

for key, expected in expected_counts.items():
    observed = str(summary_map.get(key, ""))
    assert observed == expected, f"{key}: expected {expected}, observed {observed}"

assert lock.get("status") == "FINAL_LOCKED_TEST_EVALUATED_ONCE"
assert abs(float(lock.get("threshold")) - 0.42) < 1e-12

print("Final sentiment integrity checks: PASS")
print(f"Locked-test status: {lock['status']}")
print(f"Threshold: {lock['threshold']}")

Final sentiment integrity checks: PASS
Locked-test status: FINAL_LOCKED_TEST_EVALUATED_ONCE
Threshold: 0.42


## Visual 1 ? HCC vs Non-HCC

Visual ini memakai denominator komentar **evaluable** pada masing-masing grup. `Uncertain` dan `No Text` tidak dipaksa menjadi Neutral.

In [4]:
plot_df = hcc_vs_nonhcc.set_index("group").loc[["Non-HCC", "HCC"]]
ratio_cols = ["positive_ratio_evaluable", "neutral_ratio_evaluable", "negative_ratio_evaluable"]
labels = ["Positive", "Neutral", "Negative"]

fig, ax = plt.subplots(figsize=(12, 6.8))
left = np.zeros(len(plot_df))
y = np.arange(len(plot_df))

for label, col in zip(labels, ratio_cols):
    values = plot_df[col].astype(float).to_numpy() * 100
    ax.barh(y, values, left=left, color=COLORS[label], label=label, height=0.54)
    for i, value in enumerate(values):
        if value >= 4:
            ax.text(left[i] + value / 2, i, f"{value:.1f}%", ha="center", va="center", color="white", fontsize=11)
    left += values

ax.set_yticks(y, plot_df.index)
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_xlabel("Share of evaluable comments")
ax.set_title("Final Sentiment Orientation: HCC vs Non-HCC (100% Stacked)", pad=14, fontsize=15)
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.25), frameon=True)
coverage_note = "; ".join(
    f"{idx}: coverage {row['coverage'] * 100:.1f}%, uncertain {int(row['uncertain_count'])}, no text {int(row['no_text_count'])}"
    for idx, row in plot_df.iterrows()
)
fig.text(0.01, 0.02, coverage_note, fontsize=9)
fig.tight_layout(rect=(0, 0.08, 1, 1))
out_path = VIS_DIR / "sentiment_hcc_vs_nonhcc_100pct.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(relative_to_root(out_path))

output/rm2_sentiment/final/visualisasi/sentiment_hcc_vs_nonhcc_100pct.png


## Visual 2 ? Distribusi Sentimen Observasional

In [5]:
dist_plot = distribution.loc[distribution["label"].isin(LABEL_ORDER)].copy()
dist_plot["label"] = pd.Categorical(dist_plot["label"], categories=LABEL_ORDER, ordered=True)
dist_plot = dist_plot.sort_values("label")

fig, ax = plt.subplots(figsize=(10, 6))
values = dist_plot["count"].astype(int).to_numpy()
labels_plot = dist_plot["label"].astype(str).to_list()
bars = ax.bar(labels_plot, values, color=[COLORS[label] for label in labels_plot])
for bar, (_, row) in zip(bars, dist_plot.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(row['count']):,}\n{float(row['percentage_of_total']):.1f}%",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.set_ylabel("Number of observational comments")
ax.set_title("Final Sentiment Distribution for Observational Comments", pad=14, fontsize=15)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
out_path = VIS_DIR / "sentiment_distribution_observational.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(relative_to_root(out_path))

output/rm2_sentiment/final/visualisasi/sentiment_distribution_observational.png


## Visual 3 ? Locked-Test Confusion Matrix

Confusion matrix ini berasal dari evaluasi locked test final yang sudah dilakukan satu kali dan tercatat di evaluation lock.

In [6]:
cm = confusion.set_index("true_label").loc[["Negative", "Neutral", "Positive"], ["Negative", "Neutral", "Positive"]].astype(int)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(len(cm.columns)), cm.columns)
ax.set_yticks(range(len(cm.index)), cm.index)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True human label")
ax.set_title("Final Locked-Test Confusion Matrix", pad=14, fontsize=15)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        value = cm.values[i, j]
        ax.text(j, i, str(value), ha="center", va="center", color="white" if value > cm.values.max() * 0.55 else "black", fontsize=12)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
out_path = VIS_DIR / "sentiment_validation_confusion_matrix.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(relative_to_root(out_path))

output/rm2_sentiment/final/visualisasi/sentiment_validation_confusion_matrix.png


## Visual 4 ? Goal Orientation HCC

Goal orientation dibaca sebagai orientasi pesan deskriptif berbasis pola sentimen teramati, bukan bukti niat, afiliasi, atau koordinasi komersial.

In [7]:
goal_counts = hcc_goals["goal_orientation"].value_counts().reindex(["Neutral Engagement", "Promotional / Supportive"]).dropna().astype(int)
fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.bar(goal_counts.index, goal_counts.values, color=["#8F8F8F", "#2E7D32"])
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(bar.get_height())), ha="center", va="bottom", fontsize=12)
ax.set_ylabel("Number of HCC")
ax.set_title("Final HCC Goal Orientation", pad=14, fontsize=15)
ax.grid(axis="y", alpha=0.25)
ax.set_ylim(0, max(goal_counts.values) * 1.18)
fig.tight_layout()
out_path = VIS_DIR / "hcc_goal_orientation.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(relative_to_root(out_path))

output/rm2_sentiment/final/visualisasi/hcc_goal_orientation.png


## Ringkasan Interpretasi Aman

Hasil sentimen menunjukkan orientasi pesan teramati pada komentar, bukan bukti niat, pembayaran, afiliasi, kontrol, pengaruh kausal, bot, atau astroturfing. Perbandingan HCC dan Non-HCC dipakai sebagai deskripsi distribusi pesan pada posisi jaringan yang berbeda.

In [8]:
visual_files = sorted(p.name for p in VIS_DIR.glob("*.png"))
print("Final visual files:")
for name in visual_files:
    print(f"- {name}")

print("\nKey final counts:")
for key in ["observational_Positive", "observational_Neutral", "observational_Negative", "observational_Uncertain", "observational_No Text"]:
    print(f"- {key}: {summary_map[key]}")

Final visual files:
- hcc_goal_orientation.png
- sentiment_distribution_observational.png
- sentiment_hcc_vs_nonhcc_100pct.png
- sentiment_validation_confusion_matrix.png

Key final counts:
- observational_Positive: 2718
- observational_Neutral: 23977
- observational_Negative: 4771
- observational_Uncertain: 1593
- observational_No Text: 4
